# Agent的高级用法-流式输出

## 1、values输出模式

当`stream_mode`设置为values模式时，每个步骤执行后，都会输出完整的状态信息，适用于每一步都要获取完整状态、状态持久化场景。

In [1]:
from dataclasses import dataclass

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}}
)

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15,"total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-31"
    }


# 创建客户服务Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)

for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="values"
):
    rprint(chunk)
    print("-" * 50)


## 2、updates输出模式

这种模式就是默认模式。该模式中，每个步骤执行后，只增量更新状态中发生变化的内容，用于监控Agent 执行进度，例如观察Agent决定调用工具、工具执行结果等步骤。

In [3]:
for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="updates"
):
    rprint(chunk)
    print("-" * 50)

{
    'model': {
        'messages': [
            AIMessage(
                content='我来为您查询客户 cust1234 的完整信息、历史订单和可用优惠。让我同时获取这些数据。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 123,
                        'prompt_tokens': 466,
                        'total_tokens': 589,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                        'prompt_cache_hit_tokens': 384,
                        'prompt_cache_miss_tokens': 82
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                    'id': '88906ca7-fbbd-48e3-80e2-0e6cbb3132bb',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a02384-9723-7d23-bde6-9abe4b667227-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_00_YeoNbsmad2kWICQ6yB0u0785',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_01_olGeJuxvvDQFgOatR92S2846',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_02_GQU2SCW8ivk78rP3TXpD8394',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 466,
                    'output_tokens': 123,
                    'total_tokens': 589,
                    'input_token_details': {'cache_read': 384},
                    'output_token_details': {}
                }
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='16041f8a-8189-4f3f-abf6-4c6643bd1c47',
                tool_call_id='call_00_YeoNbsmad2kWICQ6yB0u0785'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"total_orders": 15, "total_spent": 25800.0}',
                name='check_order_history',
                id='cdb10690-ddc6-4878-bf88-1e66b6ae17ff',
                tool_call_id='call_01_olGeJuxvvDQFgOatR92S2846'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
                name='get_current_promotions',
                id='24d362f4-80b1-44cb-b227-195e99680648',
                tool_call_id='call_02_GQU2SCW8ivk78rP3TXpD8394'
            )
        ]
    }
}

--------------------------------------------------


{
    'model': {
        'messages': [
            AIMessage(
                content='我已经成功查询到客户 cust1234 的全部信息，以下是汇总结果：\n\n---\n\n### 📋 
客户基本信息\n- **客户姓名**：张三\n- **客户等级**：VIP\n- **加入日期**：2023-01-15\n\n### 🛒 历史订单\n- 
**总订单数**：15 单\n- **总消费金额**：¥25,800.00\n\n### 🎁 当前可用优惠\n- **促销活动**：\n  1. 老用户优惠\n  2. 
会员专属折扣\n- **优惠有效期**：至 2027-01-31\n\n---\n\n以上就是客户 
cust1234（张三）的完整信息、历史订单数据和当前可用的优惠活动。如果您需要进一步的分析或其他帮助，请随时告诉我！',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 165,
                        'prompt_tokens': 685,
                        'total_tokens': 850,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                        'prompt_cache_hit_tokens': 384,
                        'prompt_cache_miss_tokens': 301
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                    'id': '8e4f6476-1f1f-457b-a6cd-3336582dd958',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a02384-9d42-7b82-92e0-5b36caf2fd62-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 685,
                    'output_tokens': 165,
                    'total_tokens': 850,
                    'input_token_details': {'cache_read': 384},
                    'output_token_details': {}
                }
            )
        ]
    }
}

--------------------------------------------------


## 3、messages输出模式

该模式中会输出流式返回的Token以及相关的元数据（如：来自哪个节点），可以用在实现类似 ChatGPT 的打字机效果场景，为聊天机器人等交互式应用提供最佳的实时体验。

In [4]:
for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="messages"
):
    # rprint(chunk)
    # print("-" * 50)
    print(chunk[0].content,end="",flush=True)

我来为您查询客户 cust1234 的完整信息、历史订单和当前可用优惠。我将同时发起这几个查询。{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}{"total_orders": 15, "total_spent": 25800.0}{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}我已经成功查询到客户 cust1234 的完整信息，以下是汇总结果：

---

### 📋 客户基本信息
| 项目 | 内容 |
|------|------|
| **客户ID** | cust1234 |
| **姓名** | 张三 |
| **会员等级** | VIP |
| **加入日期** | 2023-01-15 |

---

### 🛒 历史订单信息
| 项目 | 内容 |
|------|------|
| **总订单数** | 15 笔 |
| **累计消费金额** | ¥25,800 |

---

### 🎁 当前可用优惠活动
| 活动名称 | 有效期 |
|----------|--------|
| 老用户优惠 | 至 2027-01-31 |
| 会员专属折扣 | 至 2027-01-31 |

---

**总结：** 客户张三是一名 VIP 会员，自 2023 年 1 月加入以来共下单 15 笔，累计消费 ¥25,800。目前可享受"老用户优惠"和"会员专属折扣"两项促销活动，有效期均至 2027 年 1 月 31 日。

如需进一步查询或了解某项优惠的具体使用条件，请随时告诉我！